# Guidance ablations with the evaluation metrics (development pairs), unattended

Registered in `docs/ablation_guidance_sheet_2026-10-24.md` (fill the two "Registered by" boxes and push BEFORE running:
cell 3 refuses to continue while they are blank). Six settings, 33 videos, 33 frames, seed 0, on the five development
pairs that pass the camera check; the six pairs' root `motion_transfer` (the test set is never touched). Everything is
resumable: a video that exists is kept, every graded row is appended to `reports/m3_ablation/guidance.json`, the log is
`reports/m3_ablation/guidance_run.log` on Drive. If the VM disconnects, open the notebook and **Run all** again: the
finished work is skipped.

Needs an A100 runtime (Phase-2 environment, ~27 min to set up on a fresh VM). With Colab Pro, enable *Runtime >
Background execution* so the tab may close; otherwise keep the tab open. Expected time: about 3.5 h of sampling plus
grading.

When you wake: cell 5 prints the rows per setting, the failures with their reason, and the export folder for the
evaluation (`reports/m3_ablation/export_guidance/`).

In [ ]:
# 1. Drive mount, the GitHub token from Drive (never printed, never persisted into the repo), clone-or-pull of the PRIVATE repo
import os, subprocess
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
GITHUB_URL = "https://github.com/jnheinrich451-eng/mixture-motion-transfer.git"; BRANCH = "main"; REPO_DIR = Path("/content/repo")

def _load_secret(path):
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            return line.split("=", 1)[1].strip().strip('"').strip("'")
    raise ValueError(f"no KEY=VALUE line in {path}")

os.environ["GH_TOKEN"] = _load_secret("/content/drive/MyDrive/secrets/gh_token.env")
os.environ["HF_TOKEN"] = _load_secret("/content/drive/MyDrive/secrets/hf_token.env")
subprocess.run(["git", "config", "--global", "credential.helper", "store --file /tmp/.git-credentials"], check=True)
Path("/tmp/.git-credentials").write_text("https://x-access-token:" + os.environ["GH_TOKEN"] + "@github.com" + chr(10)); os.chmod("/tmp/.git-credentials", 0o600)
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "-b", BRANCH, GITHUB_URL, str(REPO_DIR)], check=True)
else:
    r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH])
    if r.returncode != 0:
        print("WARN: git pull failed: the VM copy has local edits or diverged; commit/stash them or delete /content/repo and re-run")
%cd {REPO_DIR}

In [ ]:
!cd /content/repo && git log --oneline -1

In [ ]:
# 1b. Only if cell 1 printed "WARN: git pull failed": the VM copy carries the records cell 5 copied from Drive, which are now
#     tracked files; Drive holds the originals, so discard the VM copies and pull (cell 5 copies them again at the end).
!cd /content/repo && git status --short | head -20
!cd /content/repo && git checkout -- . && git clean -fd measurements && git pull --ff-only origin main && git log --oneline -1


In [ ]:
# 2. Phase-2 environment (Wan runtime, MoGe) and the reference videos on local disk; ~27 min on a fresh VM, seconds after
!bash scripts/bootstrap_phase2.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 3. Registration check (rule 19: predictions are registered before the run). Refuses while a "Registered by" box is blank.
from pathlib import Path
sheet = Path("docs/ablation_guidance_sheet_2026-10-24.md").read_text(encoding="utf-8")
blank = [ln for ln in sheet.splitlines() if ln.startswith("Registered by:") and "______" in ln]
print(sheet[sheet.index("## Predictions"):sheet.index("## Reporting")])
assert not blank, f"{len(blank)} 'Registered by' box(es) still blank in docs/ablation_guidance_sheet_2026-10-24.md: fill, commit, push, re-run cell 1"
print("registered: OK")

In [ ]:
!cd /content/repo && python scripts/run_m3_ablation_guidance.py --euler \
    --drive-root /content/drive/MyDrive/motion_transfer --videos-dir /content/data/videos \
    2>&1 | tee -a /content/drive/MyDrive/motion_transfer/reports/m3_ablation/guidance_run.log


In [ ]:
# 4. All six settings, resumable; the log goes to Drive. Order inside the runner: per pair, then setting.
import os
os.environ["MT_ROOT"] = "/content/drive/MyDrive/motion_transfer"
os.environ["MT_DATA"] = "/content/data"
os.makedirs("/content/drive/MyDrive/motion_transfer/reports/m3_ablation", exist_ok=True)
!cd /content/repo && python scripts/run_m3_ablation_guidance.py --noobj --coff --euler --gammas 0.5 0.25 --hardmask --skyout \
    --drive-root /content/drive/MyDrive/motion_transfer --videos-dir /content/data/videos \
    2>&1 | tee -a /content/drive/MyDrive/motion_transfer/reports/m3_ablation/guidance_run.log

In [ ]:
# 5. Summary when you wake: rows per setting, failures with their reason, and what the evaluation reads
import json, shutil
from pathlib import Path
from collections import Counter
rep = Path("/content/drive/MyDrive/motion_transfer/reports/m3_ablation")
rows = json.loads((rep / "guidance.json").read_text(encoding="utf-8")) if (rep / "guidance.json").exists() else []
ok = [r for r in rows if r.get("graded_rc") == 0]; bad = [r for r in rows if r.get("graded_rc") != 0]
open_fail = [r for r in bad if (r["pair"], r["setting"]) not in {(o["pair"], o["setting"]) for o in ok}]
print(f"{len(ok)} graded rows; {len(bad)} failed attempts on record, of which {len(open_fail)} settings are still without a graded row")
print("per setting:", dict(Counter(r["setting"] for r in ok)))
print("per pair:", dict(Counter(r["pair"] for r in ok)))
for r in open_fail:
    print("STILL FAILING", r["pair"], r["setting"], "rc", r.get("rc", r.get("graded_rc")), (r.get("tail") or "")[-300:].replace(chr(10), " | "))
ix = rep / "export_guidance" / "index.json"
if ix.exists():
    idx = json.loads(ix.read_text(encoding="utf-8")); print(f"export: {len(idx)} videos in {ix.parent} (roles: {dict(Counter(r['role'] for r in idx))})")
dst = Path("/content/repo/measurements/m3_ablation"); dst.mkdir(parents=True, exist_ok=True)
for f in ("guidance.json", "guidance_run.log"):
    if (rep / f).exists(): shutil.copy2(rep / f, dst / f)
if ix.exists(): shutil.copy2(ix, dst / "export_guidance_index.json")
print("copied to measurements/m3_ablation/ in the repo (commit and push when you are back)")

## Afterwards

1. Commit `measurements/m3_ablation/guidance.json`, `guidance_run.log` and `export_guidance_index.json` and push.
2. Score `reports/m3_ablation/export_guidance/` with the evaluation (the Chapter 4 metrics), by content hash, as for the
   window ablation v2; rows that fail the tracker's consistency check are scored with the flag, as before.
3. The Chapter 4 table (the form of Table 4.12, one block per setting) and the scored predictions are then written from
   the scored rows; nothing is averaged into the test set.